# Catalog AGN Power Law, Bolometric Luminosity, And Torus Covering Fraction

Use this notebook to make redshift trends for an AGN catalog. It defaults to `table_7.csv`, but it is designed so you can replace that file with your own CSV and map your column names in the Setup cell.

At minimum, provide one row per source and an object identifier plus redshift:

- `SOURCE_ID_COL`: unique object name or ID
- `REDSHIFT_COL`: source redshift

Add these optional columns to enable more sections:

- Power-law plot: `ALPHA_LAMBDA_COL`, and optionally `ALPHA_LAMBDA_ERR_COL` and `M2500_COL`
- Variability luminosity: `LOG_SIGMA_UV_COL`, `LOG_SIGMA_UV_ERR_COL`, `LOG_TAU_UV_RF_COL`, `LOG_TAU_UV_RF_ERR_COL`
- Catalog cross-match and fitting: `RA_COL` and `DEC_COL` in decimal degrees

The notebook checks which columns contain finite values and skips sections that your catalog cannot support, so you can use it for a partial catalog without editing the plotting code.

Workflow:
- load the configured catalog and normalize its column names internally
- plot AGN power-law slope versus redshift when slope columns are available
- inspect the most extreme catalog slopes as a quick QA check
- estimate variability-based bolometric luminosity when variability columns are available
- optionally run resumable JAXSEDFit fits for torus covering fraction (`fcov`)


## Setup

Edit only this cell for a new catalog. Set `DATA_PATH` to your CSV, give the run a readable `DATASET_LABEL`, and map the variables such as `SOURCE_ID_COL`, `REDSHIFT_COL`, `RA_COL`, and `DEC_COL` to the column names in your file.

`table_7.csv` is the default example. For another catalog, the file can live anywhere, but paths relative to `ROOT` are usually easiest to keep portable. Outputs go to `notebook_outputs/12_catalog_power_lbol_fcov_<OUTPUT_TAG>`.

Column expectations:

- Object ID and redshift are the core inputs used throughout the notebook.
- RA/Dec are only needed for the optional cross-match and JAXSEDFit fitting cells.
- `alpha_lambda`-style columns are only needed for the catalog power-law plot and extreme-slope QA table.
- UV variability columns are only needed for the bolometric-luminosity estimate.
- Missing optional data is okay; those sections will print a skip message instead of failing.


In [ ]:
from pathlib import Path
import csv
import math
import os


def find_jaxsedfit_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "jaxsedfit").is_dir():
            return path
        nested = path / "jaxsedfit"
        if (nested / "src" / "jaxsedfit").is_dir():
            return nested
    raise RuntimeError("Could not find the jaxsedfit repository root")


ROOT = find_jaxsedfit_root()

# Catalog settings. Table 7 remains the default, but any CSV can be used by
# changing DATA_PATH and mapping the relevant column names below.
DATASET_LABEL = "Table 7"
DATA_PATH = ROOT / "table_7.csv"
OUTPUT_TAG = DATA_PATH.stem
OUTPUT_DIR = ROOT / "notebook_outputs" / f"12_catalog_power_lbol_fcov_{OUTPUT_TAG}"
MPLCONFIG_DIR = OUTPUT_DIR / "mplconfig"

# Required for most plots/fits.
SOURCE_ID_COL = "SDSS Name"
REDSHIFT_COL = "z"
REDSHIFT_ERR_COL = "z_err"
RA_COL = "RA"
DEC_COL = "Dec"
M2500_COL = "m_2500"

# Optional power-law slope columns.
ALPHA_LAMBDA_COL = "alpha_lambda"
ALPHA_LAMBDA_ERR_COL = "alpha_lambda_err"

# Optional variability luminosity columns.
LOG_SIGMA_UV_COL = "log_sigma_UV"
LOG_SIGMA_UV_ERR_COL = "log_sigma_UV_err"
LOG_TAU_UV_RF_COL = "log_tau_UV_RF"
LOG_TAU_UV_RF_ERR_COL = "log_tau_UV_RF_err"

# Backwards-compatible alias used by older cells in this notebook.
TABLE7_PATH = DATA_PATH

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

print(f"Repository root: {ROOT}")
print(f"Dataset: {DATASET_LABEL}")
print(f"Reading: {DATA_PATH}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def as_float(value, default=np.nan):
    if value is None or value == "":
        return default
    try:
        text = str(value).strip()
        if text == "" or text.lower() in {"nan", "none", "--"}:
            return default
        return float(text)
    except Exception:
        return default


def safe_range(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return "no finite values"
    return f"{finite.min():.3f} to {finite.max():.3f}"


def first_present(row, *columns, default=""):
    for column in columns:
        if column in row and row[column] not in (None, ""):
            return row[column]
    return default


if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Catalog not found: {DATA_PATH}")

with DATA_PATH.open(newline="") as handle:
    reader = csv.DictReader(handle)
    rows = [dict(row) for row in reader]
    fieldnames = set(reader.fieldnames or [])

for index, row in enumerate(rows):
    # Normalize configurable catalog columns onto the Table 7 field names used by
    # the older cells below. This keeps the notebook general without rewriting
    # every analysis block.
    row["SDSS Name"] = str(first_present(row, SOURCE_ID_COL, "SDSS Name", default=f"row_{index}"))
    row["z"] = first_present(row, REDSHIFT_COL, "z")
    row["z_err"] = first_present(row, REDSHIFT_ERR_COL, "z_err")
    row["RA"] = first_present(row, RA_COL, "RA")
    row["Dec"] = first_present(row, DEC_COL, "Dec")
    row["m_2500"] = first_present(row, M2500_COL, "m_2500")
    row["alpha_lambda"] = first_present(row, ALPHA_LAMBDA_COL, "alpha_lambda")
    row["alpha_lambda_err"] = first_present(row, ALPHA_LAMBDA_ERR_COL, "alpha_lambda_err")
    row["log_sigma_UV"] = first_present(row, LOG_SIGMA_UV_COL, "log_sigma_UV")
    row["log_sigma_UV_err"] = first_present(row, LOG_SIGMA_UV_ERR_COL, "log_sigma_UV_err")
    row["log_tau_UV_RF"] = first_present(row, LOG_TAU_UV_RF_COL, "log_tau_UV_RF")
    row["log_tau_UV_RF_err"] = first_present(row, LOG_TAU_UV_RF_ERR_COL, "log_tau_UV_RF_err")

catalog_rows = rows
catalog_by_id = {row["SDSS Name"]: row for row in rows}

sdss_name = np.array([row["SDSS Name"] for row in rows])
redshift = np.array([as_float(row["z"]) for row in rows])
redshift_err = np.array([as_float(row["z_err"]) for row in rows])
alpha_lambda = np.array([as_float(row["alpha_lambda"]) for row in rows])
alpha_lambda_err = np.array([as_float(row["alpha_lambda_err"]) for row in rows])
m_2500 = np.array([as_float(row["m_2500"]) for row in rows])

HAS_POWER_LAW = np.isfinite(alpha_lambda).any()
HAS_VARIABILITY_LBOL = all(
    np.isfinite([as_float(row[column]) for row in rows]).any()
    for column in ["log_sigma_UV", "log_tau_UV_RF"]
)
HAS_POSITIONS = np.isfinite([as_float(row["RA"]) for row in rows]).any() and np.isfinite([as_float(row["Dec"]) for row in rows]).any()

valid = np.isfinite(redshift) & np.isfinite(alpha_lambda)

print(f"Loaded {len(rows):,} rows from {DATASET_LABEL}")
print(f"Rows with finite redshift and alpha_lambda values: {valid.sum():,}")
print(f"Rows with finite coordinates: {sum(np.isfinite([as_float(row['RA']) for row in rows]) & np.isfinite([as_float(row['Dec']) for row in rows])):,}")
print(f"Redshift range: {safe_range(redshift)}")
print(f"alpha_lambda range: {safe_range(alpha_lambda)}")
print(f"Power-law section available: {HAS_POWER_LAW}")
print(f"Variability Lbol section available: {HAS_VARIABILITY_LBOL}")


## Plot AGN Power-Law Slope Versus Redshift

This section needs finite redshift values and a power-law slope column mapped by `ALPHA_LAMBDA_COL`. If uncertainty and 2500 Angstrom magnitude columns are available, they are used for error bars and point color. The black line shows the median `alpha_lambda` in redshift bins.


In [ ]:
plot_data = {
    "name": sdss_name[valid],
    "z": redshift[valid],
    "z_err": redshift_err[valid],
    "alpha_lambda": alpha_lambda[valid],
    "alpha_lambda_err": alpha_lambda_err[valid],
    "m_2500": m_2500[valid],
}

if len(plot_data["z"]) == 0:
    print(f"Skipping power-law plot: {DATASET_LABEL} has no finite redshift/{ALPHA_LAMBDA_COL} pairs.")
else:
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)

    err_mask = np.isfinite(plot_data["alpha_lambda_err"]) & (plot_data["alpha_lambda_err"] > 0)
    xerr = np.where(np.isfinite(plot_data["z_err"]) & (plot_data["z_err"] > 0), plot_data["z_err"], 0.0)
    ax.errorbar(
        plot_data["z"][err_mask],
        plot_data["alpha_lambda"][err_mask],
        xerr=xerr[err_mask],
        yerr=plot_data["alpha_lambda_err"][err_mask],
        fmt="none",
        ecolor="0.78",
        elinewidth=0.5,
        alpha=0.18,
        zorder=1,
    )

    scatter = ax.scatter(
        plot_data["z"],
        plot_data["alpha_lambda"],
        c=plot_data["m_2500"],
        s=13,
        cmap="viridis_r",
        alpha=0.62,
        linewidths=0,
        zorder=2,
    )

    bins = np.linspace(np.nanmin(plot_data["z"]), np.nanmax(plot_data["z"]), 13)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_medians = []
    bin_counts = []
    for left, right in zip(bins[:-1], bins[1:]):
        in_bin = (plot_data["z"] >= left) & (plot_data["z"] < right)
        bin_counts.append(in_bin.sum())
        bin_medians.append(np.nanmedian(plot_data["alpha_lambda"][in_bin]) if in_bin.any() else np.nan)

    bin_medians = np.array(bin_medians)
    bin_counts = np.array(bin_counts)
    median_mask = np.isfinite(bin_medians) & (bin_counts > 0)
    ax.plot(
        bin_centers[median_mask],
        bin_medians[median_mask],
        color="black",
        marker="o",
        markersize=4,
        linewidth=1.8,
        label="Median in redshift bins",
        zorder=3,
    )

    ax.axhline(0, color="0.35", linewidth=0.8, linestyle="--", zorder=0)
    ax.set_xlabel("Redshift, z")
    ax.set_ylabel(r"AGN power-law slope, $\alpha_\lambda$")
    ax.set_title(f"{DATASET_LABEL} AGN Power-Law Slope as a Function of Redshift")
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, loc="best")

    cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
    cbar.set_label(r"$m_{2500}$")

    figure_path = OUTPUT_DIR / f"{OUTPUT_TAG}_alpha_lambda_vs_redshift.png"
    fig.savefig(figure_path, dpi=220)
    print(f"Saved figure to {figure_path}")
    plt.show()


## Optional: Inspect The Most Extreme Slopes

This quick table is useful for catching suspicious slope measurements, bad redshifts, or catalog rows that may need cleaning before fitting.


In [ ]:
if len(plot_data["alpha_lambda"]) == 0:
    print(f"Skipping extremes table: {DATASET_LABEL} has no finite alpha_lambda values.")
else:
    order = np.argsort(plot_data["alpha_lambda"])
    n_extreme = min(5, len(order))
    extreme_indices = np.unique(np.concatenate([order[:n_extreme], order[-n_extreme:]]))

    print(f"{SOURCE_ID_COL:<20} z       alpha_lambda  alpha_lambda_err")
    print("-" * 58)
    for idx in extreme_indices:
        print(
            f"{plot_data['name'][idx]:<20} "
            f"{plot_data['z'][idx]:6.3f} "
            f"{plot_data['alpha_lambda'][idx]:13.3f} "
            f"{plot_data['alpha_lambda_err'][idx]:16.3f}"
        )


## Variability-Based Bolometric Luminosity

This section estimates `L_2500A` from UV variability quantities and applies a configurable bolometric correction. It only runs when the catalog has finite values for the mapped `LOG_SIGMA_UV_COL` and `LOG_TAU_UV_RF_COL`; uncertainty columns improve the error estimates when available.


## Estimate Luminosity From Variability Columns

The calculation uses:

`L_2500A = (9.3e44 erg/s) * (sigma_UV / 0.2 mag)^(-2.52) * (tau_UV,RF / 800 days)^(0.62)`

The default Table 7 convention stores `log_sigma_UV` and `log_tau_UV_RF` as base-10 logs of `sigma_UV` and `tau_UV,RF`. If your catalog uses different names, update the column mapping in Setup. If your catalog stores linear values instead of log values, convert them before using this section.

The notebook then applies:

`L_bol = BC_2500 * L_2500A`

The default is `BC_2500 = 5.15`; change that value in the code cell below if you want a different correction.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

L2500_NORM = 9.3e44
L2500_NORM_ERR = 0.4e44
SIGMA_UV_NORM = 0.2
TAU_UV_RF_NORM_DAYS = 800.0
SIGMA_EXPONENT = -2.52
SIGMA_EXPONENT_ERR = 0.06
TAU_EXPONENT = 0.62
TAU_EXPONENT_ERR = 0.033
BOL_CORRECTION_2500 = 5.15

sdss_name = np.array([row["SDSS Name"] for row in rows])
redshift = np.array([as_float(row["z"]) for row in rows])
log_sigma_uv = np.array([as_float(row["log_sigma_UV"]) for row in rows])
log_sigma_uv_err = np.array([as_float(row["log_sigma_UV_err"]) for row in rows])
log_tau_uv_rf = np.array([as_float(row["log_tau_UV_RF"]) for row in rows])
log_tau_uv_rf_err = np.array([as_float(row["log_tau_UV_RF_err"]) for row in rows])

if not HAS_VARIABILITY_LBOL:
    log_l2500 = np.full(len(rows), np.nan)
    log_lbol = np.full(len(rows), np.nan)
    log_lbol_err = np.full(len(rows), np.nan)
    valid = np.zeros(len(rows), dtype=bool)
    print(
        "Skipping variability-based bolometric luminosity: "
        f"{DATASET_LABEL} needs finite {LOG_SIGMA_UV_COL} and {LOG_TAU_UV_RF_COL} values."
    )
else:
    log_sigma_ratio = log_sigma_uv - np.log10(SIGMA_UV_NORM)
    log_tau_ratio = log_tau_uv_rf - np.log10(TAU_UV_RF_NORM_DAYS)

    log_l2500 = (
        np.log10(L2500_NORM)
        + SIGMA_EXPONENT * log_sigma_ratio
        + TAU_EXPONENT * log_tau_ratio
    )
    log_lbol = np.log10(BOL_CORRECTION_2500) + log_l2500

    log_l2500_err = np.sqrt(
        (L2500_NORM_ERR / (L2500_NORM * np.log(10)))**2
        + (log_sigma_ratio * SIGMA_EXPONENT_ERR)**2
        + (SIGMA_EXPONENT * log_sigma_uv_err)**2
        + (log_tau_ratio * TAU_EXPONENT_ERR)**2
        + (TAU_EXPONENT * log_tau_uv_rf_err)**2
    )
    log_lbol_err = log_l2500_err

    valid = np.isfinite(redshift) & np.isfinite(log_lbol)

    print(f"Loaded {len(rows):,} rows from {DATASET_LABEL}")
    print(f"Valid redshift and estimated logLbol values: {valid.sum():,}")
    print(f"Redshift range: {safe_range(redshift)}")
    print(f"Estimated logL2500 range: {safe_range(log_l2500)}")
    print(f"Estimated logLbol range: {safe_range(log_lbol)}")
    print(f"Using BC_2500 = {BOL_CORRECTION_2500}")


## Plot Estimated Log Bolometric Luminosity Versus Redshift

In [ ]:
plot_name = sdss_name[valid]
plot_z = redshift[valid]
plot_log_lbol = log_lbol[valid]
plot_log_lbol_err = log_lbol_err[valid]
plot_log_l2500 = log_l2500[valid]
plot_log_sigma_uv = log_sigma_uv[valid]

if len(plot_z) == 0:
    print(f"Skipping Lbol plot: {DATASET_LABEL} has no finite redshift/logLbol pairs.")
else:
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)

    err_mask = np.isfinite(plot_log_lbol_err) & (plot_log_lbol_err > 0)
    ax.errorbar(
        plot_z[err_mask],
        plot_log_lbol[err_mask],
        yerr=plot_log_lbol_err[err_mask],
        fmt="none",
        ecolor="0.78",
        elinewidth=0.5,
        alpha=0.18,
        zorder=1,
    )

    scatter = ax.scatter(
        plot_z,
        plot_log_lbol,
        c=plot_log_sigma_uv,
        s=13,
        cmap="magma_r",
        alpha=0.65,
        linewidths=0,
        zorder=2,
    )

    bins = np.linspace(np.nanmin(plot_z), np.nanmax(plot_z), 13)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_medians = []
    bin_counts = []
    for left, right in zip(bins[:-1], bins[1:]):
        in_bin = (plot_z >= left) & (plot_z < right)
        bin_counts.append(in_bin.sum())
        bin_medians.append(np.nanmedian(plot_log_lbol[in_bin]) if in_bin.any() else np.nan)

    bin_medians = np.array(bin_medians)
    bin_counts = np.array(bin_counts)
    median_mask = np.isfinite(bin_medians) & (bin_counts > 0)
    ax.plot(
        bin_centers[median_mask],
        bin_medians[median_mask],
        color="black",
        marker="o",
        markersize=4,
        linewidth=1.8,
        label="Median in redshift bins",
        zorder=3,
    )

    ax.set_xlabel("Redshift, z")
    ax.set_ylabel(r"Estimated $\log_{10}(L_{\rm bol}\,/\,\mathrm{erg\ s^{-1}})$")
    ax.set_title(f"{DATASET_LABEL} Variability-Based Bolometric Luminosity Versus Redshift")
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, loc="best")

    cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
    cbar.set_label(r"$\log_{10}(\sigma_{\rm UV} / \mathrm{mag})$")

    figure_path = OUTPUT_DIR / f"{OUTPUT_TAG}_variability_logLbol_vs_redshift.png"
    fig.savefig(figure_path, dpi=220)
    print(f"Saved figure to {figure_path}")
    plt.show()


## Optional: Inspect The Most And Least Luminous Objects

In [ ]:
if len(plot_log_lbol) == 0:
    print(f"Skipping Lbol extremes table: {DATASET_LABEL} has no finite logLbol values.")
else:
    order = np.argsort(plot_log_lbol)
    n_extreme = min(5, len(order))
    extreme_indices = np.unique(np.concatenate([order[:n_extreme], order[-n_extreme:]]))

    print(f"{SOURCE_ID_COL:<20} z       logL2500  logLbol")
    print("-" * 47)
    for idx in extreme_indices:
        print(
            f"{plot_name[idx]:<20} "
            f"{plot_z[idx]:6.3f} "
            f"{plot_log_l2500[idx]:9.3f} "
            f"{plot_log_lbol[idx]:8.3f}"
        )


## Chimera Benchmark Comparison

This comparison uses the separate `chimera_mass_retrieval_by_logLbol.csv` benchmark file. It is independent of the configured AGN catalog above and can be skipped if you only want the catalog-level plots.


In [ ]:
CHIMERA_BENCHMARK_PATH = ROOT / "chimera_mass_retrieval_by_logLbol.csv"

with CHIMERA_BENCHMARK_PATH.open(newline="") as handle:
    chimera_rows = list(csv.DictReader(handle))

chimera_object_id = np.array([row["object_id"] for row in chimera_rows])
chimera_redshift = np.array([as_float(row["redshift"]) for row in chimera_rows])
chimera_log_lbol_qso = np.array([as_float(row["logLbol_QSO"]) for row in chimera_rows])
chimera_log_lbol_weighted = np.array([as_float(row["logLbol_chimera"]) for row in chimera_rows])
chimera_qso_weight = np.array([as_float(row["chimera_QSO_weight"]) for row in chimera_rows])

chimera_valid = np.isfinite(chimera_redshift) & np.isfinite(chimera_log_lbol_qso)

print(f"Loaded {len(chimera_rows):,} Chimera benchmark rows")
print(f"Valid redshift and logLbol_QSO values: {chimera_valid.sum():,}")
print(f"Redshift range: {np.nanmin(chimera_redshift):.3f} to {np.nanmax(chimera_redshift):.3f}")
print(f"logLbol_QSO range: {np.nanmin(chimera_log_lbol_qso):.3f} to {np.nanmax(chimera_log_lbol_qso):.3f}")
print(f"logLbol_chimera range: {np.nanmin(chimera_log_lbol_weighted):.3f} to {np.nanmax(chimera_log_lbol_weighted):.3f}")

In [ ]:
chimera_plot_z = chimera_redshift[chimera_valid]
chimera_plot_log_lbol_qso = chimera_log_lbol_qso[chimera_valid]
chimera_plot_log_lbol_weighted = chimera_log_lbol_weighted[chimera_valid]
chimera_plot_qso_weight = chimera_qso_weight[chimera_valid]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True, constrained_layout=True)

scatter = axes[0].scatter(
    chimera_plot_z,
    chimera_plot_log_lbol_qso,
    c=chimera_plot_qso_weight,
    s=13,
    cmap="plasma",
    alpha=0.65,
    linewidths=0,
)
axes[0].set_title("Intrinsic AGN bolometric luminosity")
axes[0].set_ylabel(r"$\log_{10}(L_{\rm bol,QSO})$")

axes[1].scatter(
    chimera_plot_z,
    chimera_plot_log_lbol_weighted,
    c=chimera_plot_qso_weight,
    s=13,
    cmap="plasma",
    alpha=0.65,
    linewidths=0,
)
axes[1].set_title("Chimera-weighted AGN bolometric luminosity")
axes[1].set_ylabel(r"$\log_{10}(L_{\rm bol,Chimera})$")

for ax in axes:
    ax.set_xlabel("Redshift")
    ax.grid(alpha=0.22)

cbar = fig.colorbar(scatter, ax=axes, pad=0.015)
cbar.set_label("Chimera QSO weight")

chimera_comparison_path = OUTPUT_DIR / "chimera_benchmark_logLbol_vs_redshift.png"
fig.savefig(chimera_comparison_path, dpi=220)
print(f"Saved figure to {chimera_comparison_path}")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.2), constrained_layout=True)
ax.scatter(
    chimera_plot_z,
    chimera_plot_log_lbol_qso,
    s=15,
    color="#20639B",
    alpha=0.65,
    linewidths=0,
)
ax.set_xlabel("Redshift")
ax.set_ylabel(r"$\log_{10}(L_{\rm bol,QSO})$")
ax.set_title("Chimera Benchmark Log Bolometric Luminosity Versus Redshift")
ax.grid(alpha=0.22)

chimera_single_panel_path = OUTPUT_DIR / "chimera_benchmark_logLbol_QSO_vs_redshift.png"
fig.savefig(chimera_single_panel_path, dpi=220)
print(f"Saved figure to {chimera_single_panel_path}")
plt.show()

## Torus Covering Fraction Fitting

This section plots existing `fcov` results when available and can run resumable JAXSEDFit fits to create or update the `fcov` result table.

For new fits, your catalog needs source IDs, redshifts, and sky coordinates in decimal degrees. The fitting cells query broadband photometry from the coordinates in one large chunk by default, so use a small `FCOV_MAX_OBJECTS` value for a smoke test before scaling up.


In [ ]:
FCOV_OUTPUT_DIR = OUTPUT_DIR / "fcov_fits"
FCOV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FCOV_RESULTS_PATH = FCOV_OUTPUT_DIR / "table7_fcov_fit_results.ecsv"
FCOV_FAILURES_PATH = FCOV_OUTPUT_DIR / "table7_fcov_fit_failures.ecsv"
FCOV_FIGURE_PATH = FCOV_OUTPUT_DIR / "table7_fcov_vs_redshift.png"

print(f"fcov results path: {FCOV_RESULTS_PATH}")
print(f"fcov failures path: {FCOV_FAILURES_PATH}")

## Load Configured Catalog For `fcov`

This reloads the normalized catalog rows created in Setup and selects objects with finite redshift. The optional fitting cells below will additionally require finite RA/Dec for each target.


In [ ]:
def as_float(value, default=np.nan):
    try:
        text = str(value).strip()
        if text == "" or text.lower() in {"nan", "none", "--"}:
            return default
        return float(text)
    except Exception:
        return default


table7_rows = catalog_rows
table7_by_name = catalog_by_id
sdss_name = np.array([row["SDSS Name"] for row in table7_rows], dtype=str)
redshift = np.array([as_float(row["z"]) for row in table7_rows], dtype=float)
redshift_err = np.array([as_float(row["z_err"]) for row in table7_rows], dtype=float)
m_2500 = np.array([as_float(row["m_2500"]) for row in table7_rows], dtype=float)

valid_table7 = np.isfinite(redshift)
print(f"Loaded {len(table7_rows):,} rows from {DATASET_LABEL}")
print(f"Rows with finite redshift: {valid_table7.sum():,}")


## Plot Existing `fcov` Results

If a previous fitting run has produced an ECSV result table, this cell plots covering fraction versus redshift. If no result table exists yet, run the optional fitting section below.


In [ ]:
def finite_column(table, name, default=np.nan):
    if name in table.colnames:
        return np.asarray(table[name], dtype=float)
    return np.full(len(table), default, dtype=float)


def catalog_lookup_float(source_ids, column):
    return np.array([as_float(table7_by_name.get(str(source_id), {}).get(column, np.nan)) for source_id in source_ids], dtype=float)


def plot_fcov_vs_redshift(results, output_path=FCOV_FIGURE_PATH, show=True):
    source_ids = np.asarray(results["source_id"], dtype=str)
    z = finite_column(results, "redshift")
    z_err = finite_column(results, "redshift_err")
    if not np.isfinite(z_err).any():
        z_err = catalog_lookup_float(source_ids, "z_err")
    z_err = np.where(np.isfinite(z_err) & (z_err > 0), z_err, 0.0)

    fcov = finite_column(results, "fcov_median")
    fcov_lo = finite_column(results, "fcov_p16")
    fcov_hi = finite_column(results, "fcov_p84")
    if not np.isfinite(fcov).any() and "fcov" in results.colnames:
        fcov = finite_column(results, "fcov")
    yerr = None
    if np.isfinite(fcov_lo).any() and np.isfinite(fcov_hi).any():
        yerr = np.vstack([
            np.maximum(fcov - fcov_lo, 0.0),
            np.maximum(fcov_hi - fcov, 0.0),
        ])

    color = finite_column(results, "hot_fcov_median")
    color_label = r"hot dust $f_{cov}$"
    if not np.isfinite(color).any():
        color = catalog_lookup_float(source_ids, "m_2500")
        color_label = r"$m_{2500}$"

    finite = np.isfinite(z) & np.isfinite(fcov)
    if not finite.any():
        raise ValueError("No finite fcov/redshift pairs were found in the result table.")

    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    ax.errorbar(
        z[finite],
        fcov[finite],
        xerr=z_err[finite],
        yerr=yerr[:, finite] if yerr is not None else None,
        fmt="none",
        ecolor="0.72",
        elinewidth=0.7,
        alpha=0.55,
        zorder=1,
    )
    scatter = ax.scatter(
        z[finite],
        fcov[finite],
        c=color[finite] if np.isfinite(color[finite]).any() else None,
        s=24,
        cmap="viridis",
        alpha=0.78,
        linewidths=0,
        zorder=2,
    )

    if finite.sum() >= 5:
        bins = np.linspace(np.nanmin(z[finite]), np.nanmax(z[finite]), 13)
        centers = 0.5 * (bins[:-1] + bins[1:])
        medians = []
        counts = []
        for left, right in zip(bins[:-1], bins[1:]):
            in_bin = (z >= left) & (z < right) & finite
            counts.append(in_bin.sum())
            medians.append(np.nanmedian(fcov[in_bin]) if in_bin.any() else np.nan)
        medians = np.asarray(medians)
        counts = np.asarray(counts)
        good_bins = np.isfinite(medians) & (counts > 0)
        ax.plot(
            centers[good_bins],
            medians[good_bins],
            color="black",
            marker="o",
            linewidth=1.8,
            markersize=4,
            label="Median in redshift bins",
            zorder=3,
        )
        ax.legend(frameon=False, loc="best")

    ax.set_xlabel("Redshift, z")
    ax.set_ylabel(r"AGN torus covering fraction, $f_{cov}$")
    ax.set_title(f"{DATASET_LABEL} JAXSEDFit Torus Covering Fraction Versus Redshift")
    ax.set_ylim(0.0, max(1.0, np.nanmax(fcov[finite]) * 1.08))
    ax.grid(alpha=0.22)
    if np.isfinite(color[finite]).any():
        cbar = fig.colorbar(scatter, ax=ax, pad=0.015)
        cbar.set_label(color_label)

    fig.savefig(output_path, dpi=220)
    print(f"Plotted {finite.sum():,} fitted catalog sources")
    print(f"Saved figure to {output_path}")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


if FCOV_RESULTS_PATH.exists():
    fcov_results = Table.read(FCOV_RESULTS_PATH, format="ascii.ecsv")
    print(f"Loaded {len(fcov_results):,} fcov fit rows from {FCOV_RESULTS_PATH}")
    plot_fcov_vs_redshift(fcov_results)
else:
    print(f"No fcov result table found yet at {FCOV_RESULTS_PATH}")
    print("Run the optional fitting section below to create one.")


## Optional Resumable Fits

Set `RUN_CATALOG_FCOV_FITS = True` to query broadband photometry for the configured catalog sources, fit each source with JAXSEDFit, save `fcov` and `hot_fcov`, and then re-run the plot above.

Recommended first run:

- keep `FCOV_MAX_OBJECTS` small, for example 5 to 25
- leave `FCOV_BATCH_SIZE = None` for one big chunk
- check that the queried photometry looks reasonable
- inspect failures before running the full catalog

The output tables are resumable, so interrupted runs can continue without refitting successful sources.


In [ ]:
RUN_CATALOG_FCOV_FITS = True

FCOV_MAX_OBJECTS = 25
# None means query and fit all pending targets in one big chunk.
# Set an integer only if you need to split a very large run.
FCOV_BATCH_SIZE = None
FCOV_FIT_STEPS = 800
FCOV_FIT_LEARNING_RATE = 5e-3
FCOV_MIN_BANDS_TO_FIT = 5
FCOV_MAX_MAG_ERR = 1.0
FCOV_RANDOM_SEED = 20260601

FILTER_SPECLITE_NAME = {
    "FUV_galex": "galex-fuv",
    "NUV_galex": "galex-nuv",
    "u_sdss": "sdss2010-u",
    "g_sdss": "sdss2010-g",
    "r_sdss": "sdss2010-r",
    "i_sdss": "sdss2010-i",
    "z_sdss": "sdss2010-z",
    "W1": "wise2010-W1",
    "W2": "wise2010-W2",
    "W3": "wise2010-W3",
    "W4": "wise2010-W4",
}
FILTER_ORDER = {name: i for i, name in enumerate(FILTER_SPECLITE_NAME)}


In [ ]:
if RUN_CATALOG_FCOV_FITS:
    import gc
    import astropy.units as u
    from astropy.coordinates import SkyCoord
    try:
        import jax
    except Exception:
        jax = None

    for path in (ROOT / "src", ROOT.parent / "bandwagon" / "src"):
        if path.is_dir() and str(path) not in sys.path:
            sys.path.insert(0, str(path))

    from bandwagon import matches_to_photometry, xmatch_catalogs
    from jaxsedfit.config import (
        AGNConfig,
        FilterSet,
        FitConfig,
        GalaxyConfig,
        InferenceConfig,
        LikelihoodConfig,
        Observation,
        PhotometryData,
    )
    from jaxsedfit.core import JAXSEDFit
    from jaxsedfit.filters import load_filter_curves

    dsps_ssp_fn = ROOT / "tempdata.h5"
    assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"

    def read_rows(path):
        if not path.exists():
            return []
        return [dict(row) for row in Table.read(path, format="ascii.ecsv")]

    def write_rows(rows_to_write, path):
        if rows_to_write:
            tmp_path = path.with_suffix(path.suffix + ".tmp")
            Table(rows=rows_to_write).write(tmp_path, format="ascii.ecsv", overwrite=True)
            tmp_path.replace(path)

    def save_progress():
        write_rows(fcov_success_rows, FCOV_RESULTS_PATH)
        write_rows(fcov_failure_rows, FCOV_FAILURES_PATH)

    def cleanup_memory():
        gc.collect()
        if jax is not None:
            try:
                jax.clear_caches()
            except Exception:
                pass

    def scalar_float(value, default=np.nan):
        try:
            arr = np.asarray(value, dtype=float).reshape(-1)
            return float(arr[0]) if arr.size else default
        except Exception:
            return as_float(value, default=default)

    def rows_for_source(photometry_table, source_id):
        mask = np.asarray(photometry_table["source_id"], dtype=str) == str(source_id)
        source_rows = photometry_table[mask]
        order = np.argsort([FILTER_ORDER.get(str(name), 999) for name in source_rows["filter_name"]])
        return source_rows[order]

    def build_fit_config(source_id, source_rows):
        table_row = table7_by_name[str(source_id)]
        fluxes = np.asarray(source_rows["flux_mjy"], dtype=float)
        errors = np.asarray(source_rows["flux_err_mjy"], dtype=float)
        errors = np.maximum(errors, 0.03 * fluxes)
        psf = [as_float(value) for value in source_rows["psf_fwhm_arcsec"]] if "psf_fwhm_arcsec" in source_rows.colnames else [np.nan] * len(source_rows)

        return FitConfig(
            observation=Observation(
                object_id=str(source_id),
                redshift=as_float(table_row["z"]),
                redshift_mode="fixed",
                ra=as_float(table_row["RA"]),
                dec=as_float(table_row["Dec"]),
            ),
            photometry=PhotometryData(
                filter_names=[str(name) for name in source_rows["filter_name"]],
                fluxes=fluxes.tolist(),
                errors=errors.tolist(),
                is_upper_limit=[False] * len(source_rows),
                psf_fwhm_arcsec=psf,
            ),
            filters=FilterSet(curves=load_filter_curves(filter_names)),
            galaxy=GalaxyConfig(dsps_ssp_fn=str(dsps_ssp_fn), n_wave=768),
            agn=AGNConfig(agn_type=1),
            likelihood=LikelihoodConfig(
                systematics_width=0.08,
                variability_uncertainty=True,
                use_host_capture_model=True,
            ),
            inference=InferenceConfig(
                map_steps=FCOV_FIT_STEPS,
                learning_rate=FCOV_FIT_LEARNING_RATE,
                seed=FCOV_RANDOM_SEED,
            ),
            prior_config={
                "log_stellar_mass": {"loc": 10.5, "scale": 1.25},
                "fracAGN_5100": {"loc": 0.75, "scale": 0.2},
                "ebv_gal": {"scale": 0.2},
                "ebv_agn": {"scale": 0.2},
            },
        )


In [ ]:
if RUN_CATALOG_FCOV_FITS:
    fcov_success_rows = read_rows(FCOV_RESULTS_PATH)
    fcov_failure_rows = read_rows(FCOV_FAILURES_PATH)
    completed_ids = {str(row["source_id"]) for row in fcov_success_rows}
    failed_ids = {str(row["source_id"]) for row in fcov_failure_rows}
    attempted_ids = completed_ids | failed_ids

    target_indices = np.flatnonzero(valid_table7)
    if FCOV_MAX_OBJECTS is not None:
        target_indices = target_indices[: int(FCOV_MAX_OBJECTS)]
    target_rows = [table7_rows[int(idx)] for idx in target_indices]
    pending_rows = [row for row in target_rows if str(row["SDSS Name"]) not in attempted_ids]

    print(f"Target {DATASET_LABEL} objects: {len(target_rows):,}")
    print(f"Already successful: {len(completed_ids):,}")
    print(f"Already failed/skipped: {len(failed_ids):,}")
    print(f"Pending: {len(pending_rows):,}")


In [ ]:
if RUN_CATALOG_FCOV_FITS:
    fcov_chunk_size = len(pending_rows) if FCOV_BATCH_SIZE is None else int(FCOV_BATCH_SIZE)
    fcov_chunk_size = max(1, fcov_chunk_size)
    print(f"Fcov fitting chunk size: {fcov_chunk_size:,} pending object(s)")

    for batch_start in range(0, len(pending_rows), fcov_chunk_size):
        batch_rows = pending_rows[batch_start : batch_start + fcov_chunk_size]
        batch_ids = np.array([str(row["SDSS Name"]) for row in batch_rows], dtype=str)
        batch_ra = np.array([as_float(row["RA"]) for row in batch_rows], dtype=float)
        batch_dec = np.array([as_float(row["Dec"]) for row in batch_rows], dtype=float)
        finite_coords = np.isfinite(batch_ra) & np.isfinite(batch_dec)

        if not finite_coords.any():
            for source_id in batch_ids:
                fcov_failure_rows.append({"source_id": str(source_id), "reason": "missing RA/Dec"})
            save_progress()
            cleanup_memory()
            continue

        query_ids = batch_ids[finite_coords]
        query_coords = SkyCoord(ra=batch_ra[finite_coords] * u.deg, dec=batch_dec[finite_coords] * u.deg, frame="icrs")
        print(f"Chunk {batch_start // fcov_chunk_size + 1}: querying and fitting {len(query_ids)} objects")

        try:
            batch_matches = xmatch_catalogs(query_coords, source_id=query_ids)
            batch_photometry = matches_to_photometry(batch_matches, max_mag_err=FCOV_MAX_MAG_ERR)
        except Exception as exc:
            for source_id in query_ids:
                fcov_failure_rows.append({"source_id": str(source_id), "reason": f"xmatch failed: {exc}"})
            save_progress()
            cleanup_memory()
            continue

        for source_id in query_ids:
            try:
                source_rows = rows_for_source(batch_photometry, source_id) if len(batch_photometry) else batch_photometry
                if len(source_rows) < FCOV_MIN_BANDS_TO_FIT:
                    fcov_failure_rows.append({"source_id": str(source_id), "reason": f"only {len(source_rows)} usable bands"})
                else:
                    cfg = build_fit_config(source_id, source_rows)
                    fitter = JAXSEDFit(cfg)
                    map_result = fitter.fit_map(
                        steps=FCOV_FIT_STEPS,
                        learning_rate=FCOV_FIT_LEARNING_RATE,
                        progress_bar=False,
                    )
                    med = fitter.map_result["median"]
                    table_row = table7_by_name[str(source_id)]
                    losses = np.asarray(map_result.get("losses", [np.nan]), dtype=float)
                    fcov_success_rows.append({
                        "source_id": str(source_id),
                        "redshift": as_float(table_row["z"]),
                        "redshift_err": as_float(table_row["z_err"]),
                        "m_2500": as_float(table_row["m_2500"]),
                        "n_bands": len(source_rows),
                        "fcov_median": scalar_float(med.get("fcov", np.nan)),
                        "hot_fcov_median": scalar_float(med.get("hot_fcov", np.nan)),
                        "log_agn_amp_map": scalar_float(med.get("log_agn_amp", np.nan)),
                        "final_loss": float(losses[-1]) if losses.size else np.nan,
                    })
                save_progress()
                print(f"Saved progress after {source_id}: {len(fcov_success_rows):,} successes, {len(fcov_failure_rows):,} failures/skips")
            except Exception as exc:
                fcov_failure_rows.append({"source_id": str(source_id), "reason": f"fit failed: {exc}"})
                save_progress()
            finally:
                cleanup_memory()

        cleanup_memory()


In [ ]:
if FCOV_RESULTS_PATH.exists():
    fcov_results = Table.read(FCOV_RESULTS_PATH, format="ascii.ecsv")
    plot_fcov_vs_redshift(fcov_results)
else:
    print(f"No fcov result table found yet at {FCOV_RESULTS_PATH}")